In [1]:
from fit import *          # TrajDataset, embeddings, TimeVaryingOU, paths (and utils via fit)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Trajectory-batched dataset from fit.py: the (lead, ch, x, y) grid is built once,
# z_prev / z_next are (n_traj, P) and the loss broadcasts over trajectories.
train_ds = TrajDataset(invert_euler960, split='train', device=device)   # condition=False: x0 in each batch is None

# test code
lead, ch, x, y, z_prev, z_next, _ = next(train_ds.iterate_batches())
print(train_ds.n, 'trajectories,', train_ds.P, 'points each')
print(lead.shape, ch.shape, x.shape, y.shape, z_prev.shape, z_next.shape)
print(f"x max: {torch.max(x)}, x min {torch.min(x)}")
print(f"y max: {torch.max(y)}, y min {torch.min(y)}")
print(f"lead max: {torch.max(lead)}, lead min {torch.min(lead)}")

960 trajectories, 188416 points each
torch.Size([188416]) torch.Size([188416]) torch.Size([188416]) torch.Size([188416]) torch.Size([8, 188416]) torch.Size([8, 188416])
x max: 63.0, x min 0.0
y max: 63.0, y min 0.0
lead max: 24.0, lead min 2.0


In [3]:
val_ds = TrajDataset(invert_rk4150, split='val', device=device)
print(val_ds.n, 'trajectories,', val_ds.size, 'loss terms')

30 trajectories, 5652480 loss terms


In [4]:
# Model: TimeVaryingOU from fit.py with the best x0-free setting found in time-varying.ipynb:
#   MLP with 1 hidden layer (256 units), lead -> positional embedding, channel -> nn.Embedding,
#   x ignored (x_dim=0), y -> one Fourier mode cos/sin(2*pi*y/64) (y_dim=2).
# It returns (sigma, c2). This notebook learns c2 only (z treated as unit variance), so the sigma head is unused.
model = TimeVaryingOU(emb_dim=64, hidden_dim=256, n_hidden=1, x_dim=0, y_dim=2)
print(sum(p.numel() for p in model.parameters()), 'parameters')

34178 parameters


In [5]:
# c2-only MLE: z is assumed to have unit variance, so only alpha = exp(-c2/2) is learned.
# Same formula as before; 1 - alpha^2 is computed as -expm1(-c2), which is stable when alpha -> 1.
def c2_mle(batch, model, device):
    lead, ch, x, y, z_prev, z_next = (b.to(device) for b in batch[:-1])   # last item is x0 (None here)
    _, c2 = model(lead, x, y, ch)   # c^2 on interval [k-1, k], shape (P,)
    alpha = torch.exp(-0.5 * c2)  # dt = 1 h
    one_m_a2 = -torch.expm1(-c2)  # 1 - alpha^2
    loss = 0.5 * (torch.log(one_m_a2) + (z_next - alpha * z_prev)**2 / one_m_a2)   # (n_traj, P)
    return loss.mean(), loss.numel()

In [6]:
def c2_train(model, train_ds, val_ds, device, epochs=150, lr=3e-4, weight_decay=1e-4, cosine=True, print_every=10,
             patience=20, min_delta=1e-5):
    # Early stop: quit when val nll has not improved by more than min_delta for `patience` epochs
    # (patience=None turns it off). The best state is loaded back into the model at the end.
    # Same loop as sigma_c2_train in fit.py (AdamW, cosine schedule stepped per batch, keeps the best state),
    # with the c2-only loss. One epoch = n_traj / traj_per_batch optimizer steps, hence many epochs.
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps_per_epoch = math.ceil(train_ds.n / train_ds.traj_per_batch)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * steps_per_epoch) if cosine else None

    history = {'train': [], 'val': []}
    best = (float('inf'), None)
    wait = 0                                                     # epochs since the last val improvement
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for batch in train_ds.iterate_batches():
            loss, n = c2_mle(batch, model, device)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            total += loss.item() * n
        history['train'].append(total / train_ds.size)

        model.eval()
        total = 0.0
        with torch.no_grad():
            for batch in val_ds.iterate_batches():
                loss, n = c2_mle(batch, model, device)
                total += loss.item() * n
        history['val'].append(total / val_ds.size)
        if history['val'][-1] < best[0] - min_delta:
            best = (history['val'][-1], {k: v.detach().clone() for k, v in model.state_dict().items()})
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % print_every == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"epoch {epoch + 1}/{epochs}  train nll {history['train'][-1]:.4f}  val nll {history['val'][-1]:.4f}")
        if patience is not None and wait >= patience:
            print(f"early stop at epoch {epoch + 1}: no val improvement for {patience} epochs")
            break
    print(f"best val nll {best[0]:.5f} at epoch {int(np.argmin(history['val'])) + 1}, last val nll {history['val'][-1]:.5f}")
    history['best_state'] = best[1]
    model.load_state_dict(best[1])                               # keep the best model, not the last one
    return history


history = c2_train(model, train_ds, val_ds, device, epochs=150, lr=3e-4)

epoch 1/150  train nll 0.0764  val nll 0.1040
epoch 10/150  train nll 0.0755  val nll 0.1046
epoch 20/150  train nll 0.0754  val nll 0.1043
epoch 30/150  train nll 0.0754  val nll 0.1042
early stop at epoch 34: no val improvement for 20 epochs
best val nll 0.10368 at epoch 14, last val nll 0.10499
